In [1]:
import pandas as pd
import numpy as np
import os
import pytz

INPUT_FILE = "databento.csv"
OUTPUT_FILE = "nq_final.csv"
CHUNKSIZE = 100000

print(f"\n{'='*70}\n  STAGE 1: GENERATE CONTINUOUS CONTRACT (nq_final.csv)\n{'='*70}")

# 1. Dynamically detect time column name to prevent KeyError
sample = pd.read_csv(INPUT_FILE, nrows=5)
time_col = "datetime" if "datetime" in sample.columns else "ts_event"
print(f"  --> Detected timestamp column: '{time_col}'")

buffer = []

# 2. Stream and clean raw Databento chunks
for i, chunk in enumerate(pd.read_csv(
    INPUT_FILE,
    usecols=[
        time_col,
        "symbol",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ],
    chunksize=CHUNKSIZE
)):
    print(f"  Processing chunk {i+1}...")
    
    # Standardize time column to datetime
    chunk["datetime"] = pd.to_datetime(chunk[time_col], utc=True)
    if time_col != "datetime":
        chunk.drop(columns=[time_col], inplace=True)

    # Remove spread contracts (e.g. NQH5-NQM5)
    chunk = chunk[~chunk["symbol"].str.contains("-", regex=False)]

    # Keep only NQ contracts
    chunk = chunk[chunk["symbol"].str.startswith("NQ")]
    
    # Enforce float32/int32 to save RAM
    for col in ("open", "high", "low", "close"):
        chunk[col] = chunk[col].astype(np.float32)
    chunk["volume"] = chunk["volume"].astype(np.int32)

    buffer.append(chunk)

print("  Combining chunks...")
df = pd.concat(buffer, ignore_index=True)
del buffer  # free memory
print(f"  Total raw rows loaded: {len(df):,}")

# 3. Determine the Dominant Contract per DAY (Locks symbol to prevent intraday jumps)
df["date_temp"] = df["datetime"].dt.date
print("  Determining daily contract dominance...")
daily_vol = df.groupby(["date_temp", "symbol"], observed=True)["volume"].sum().reset_index()

# Find the contract with the absolute highest volume for each calendar day
dominant_contracts = daily_vol.sort_values("volume", ascending=False).drop_duplicates("date_temp")
date_to_active = dict(zip(dominant_contracts["date_temp"], dominant_contracts["symbol"]))

# Map the active symbol back to the main dataframe
df["active_symbol"] = df["date_temp"].map(date_to_active)

# 4. Separate active contract wicks and fallback wicks
df_active = df[df["symbol"] == df["active_symbol"]].copy()
df_alt = df[df["symbol"] != df["active_symbol"]].copy()

# 5. Locate missing minutes in the active contract
df_active_indexed = df_active.set_index("datetime")
df_alt_indexed = df_alt.set_index("datetime")
missing_times = df_alt_indexed.index.difference(df_active_indexed.index)

df_missing_fills = df_alt_indexed.loc[missing_times].copy()

if not df_missing_fills.empty:
    # Keep only the highest-volume alternative bar per missing minute
    df_missing_fills = df_missing_fills.reset_index()
    df_missing_fills.sort_values("volume", ascending=False, inplace=True)
    df_missing_fills.drop_duplicates("datetime", inplace=True)
    
    print(f"  Stitching: Identified {len(df_missing_fills):,} missing minutes to resolve.")
    
    # Calculate daily median close to establish the basis spread
    daily_medians = df.groupby(["date_temp", "symbol"], observed=True)["close"].median().unstack()
    medians_dict = daily_medians.stack().to_dict()
    
    # Create mapping keys
    df_missing_fills["active_key"] = list(zip(df_missing_fills["date_temp"], df_missing_fills["active_symbol"]))
    df_missing_fills["fallback_key"] = list(zip(df_missing_fills["date_temp"], df_missing_fills["symbol"]))
    
    # Map the medians and calculate shifts
    active_medians = df_missing_fills["active_key"].map(medians_dict)
    alt_medians = df_missing_fills["fallback_key"].map(medians_dict)
    shifts = (active_medians - alt_medians).fillna(0.0)
    
    # Shift alternative wicks into the active price bracket to eliminate the gap
    for col in ["open", "high", "low", "close"]:
        df_missing_fills[col] = (df_missing_fills[col] + shifts).astype(np.float32)
        
    df_missing_fills["symbol"] = df_missing_fills["active_symbol"]
    
    # Clean up temporary columns
    df_missing_fills.drop(columns=["date_temp", "active_key", "fallback_key", "active_symbol"], inplace=True)
    df_active.drop(columns=["date_temp", "active_symbol"], inplace=True)
    
    # Concatenate active contract data with shifted fallback data
    continuous = pd.concat([df_active, df_missing_fills], ignore_index=True)
else:
    df_active.drop(columns=["date_temp", "active_symbol"], inplace=True)
    continuous = df_active

# Clean and sort
continuous.sort_values("datetime", inplace=True)
continuous = continuous[["datetime", "symbol", "open", "high", "low", "close", "volume"]]
continuous.reset_index(drop=True, inplace=True)

# Save result
print(f"  Saving output to: {OUTPUT_FILE}")
continuous.to_csv(OUTPUT_FILE, index=False)

print(f"\n{'='*70}\n  STAGE 1 COMPLETE: Clean continuous contract generated!\n{'='*70}\n")
print(continuous["symbol"].value_counts())


  STAGE 1: GENERATE CONTINUOUS CONTRACT (nq_final.csv)
  --> Detected timestamp column: 'datetime'
  Processing chunk 1...
  Processing chunk 2...
  Processing chunk 3...
  Processing chunk 4...
  Processing chunk 5...
  Processing chunk 6...
  Processing chunk 7...
  Processing chunk 8...
  Processing chunk 9...
  Processing chunk 10...
  Processing chunk 11...
  Processing chunk 12...
  Processing chunk 13...
  Processing chunk 14...
  Processing chunk 15...
  Processing chunk 16...
  Processing chunk 17...
  Processing chunk 18...
  Processing chunk 19...
  Processing chunk 20...
  Processing chunk 21...
  Processing chunk 22...
  Processing chunk 23...
  Processing chunk 24...
  Processing chunk 25...
  Processing chunk 26...
  Processing chunk 27...
  Processing chunk 28...
  Processing chunk 29...
  Processing chunk 30...
  Processing chunk 31...
  Processing chunk 32...
  Processing chunk 33...
  Processing chunk 34...
  Processing chunk 35...
  Processing chunk 36...
  Process